# ANN Regression - Battery Health Prediction
Predicting the remaining battery capacity (%) based on usage and environmental conditions using an Artificial Neural Network.

### 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow:', tf.__version__)
print('NumPy     :', np.__version__)

### 2. Create Custom Dataset
Battery degradation dataset with 1200 samples.

Features:
- `charge_cycles` - number of times the battery has been fully charged
- `avg_temperature_c` - average operating temperature in Celsius
- `depth_of_discharge` - average DoD per cycle (0 to 1)
- `avg_charge_rate_c` - C-rate at which battery is charged (0.5 to 3.0)
- `age_months` - battery age in months
- `internal_resistance_mohm` - internal resistance in milli-ohms (increases with age)
- `avg_discharge_current_a` - average discharge current in amperes

Target:
- `capacity_remaining_pct` - remaining capacity as percentage of original (regression target)

In [ ]:
np.random.seed(7)
n = 1200

charge_cycles             = np.random.randint(10, 1500, n)
avg_temperature_c         = np.random.uniform(15, 55, n)
depth_of_discharge        = np.random.uniform(0.2, 1.0, n)
avg_charge_rate_c         = np.random.uniform(0.5, 3.0, n)
age_months                = np.random.randint(1, 60, n)
internal_resistance_mohm  = 10 + 0.05 * charge_cycles + np.random.normal(0, 2, n)
avg_discharge_current_a   = np.random.uniform(0.5, 5.0, n)

# physics-inspired capacity formula
capacity_remaining_pct = (
    100
    - 0.025  * charge_cycles
    - 0.30   * avg_temperature_c
    - 5.0    * depth_of_discharge
    - 2.5    * avg_charge_rate_c
    - 0.15   * age_months
    - 0.10   * internal_resistance_mohm
    - 0.50   * avg_discharge_current_a
    + np.random.normal(0, 1.5, n)
).clip(20, 100)                   # capacity can't go below 20% or above 100%

df = pd.DataFrame({
    'charge_cycles'            : charge_cycles,
    'avg_temperature_c'        : avg_temperature_c.round(2),
    'depth_of_discharge'       : depth_of_discharge.round(3),
    'avg_charge_rate_c'        : avg_charge_rate_c.round(2),
    'age_months'               : age_months,
    'internal_resistance_mohm' : internal_resistance_mohm.round(2),
    'avg_discharge_current_a'  : avg_discharge_current_a.round(2),
    'capacity_remaining_pct'   : capacity_remaining_pct.round(2)
})

print('Dataset shape:', df.shape)
df.head(10)

### 3. Exploratory Data Analysis

In [ ]:
print('--- Statistical Summary ---')
df.describe().round(2)

In [ ]:
print('Missing values:')
print(df.isnull().sum())

In [ ]:
# target distribution
plt.figure(figsize=(7, 4))
plt.hist(df['capacity_remaining_pct'], bins=40, color='steelblue', edgecolor='white')
plt.xlabel('Capacity Remaining (%)')
plt.ylabel('Count')
plt.title('Distribution of Target Variable')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
features = [c for c in df.columns if c != 'capacity_remaining_pct']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Each Feature vs Capacity Remaining (%)', fontsize=12)

for ax, feat in zip(axes.flat, features):
    ax.scatter(df[feat], df['capacity_remaining_pct'], alpha=0.25, s=8, color='darkcyan')
    ax.set_xlabel(feat, fontsize=8)
    ax.set_ylabel('capacity %', fontsize=8)
    ax.set_title(feat, fontsize=9)
    ax.grid(True, alpha=0.2)

# hide empty subplot
axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

### 4. Data Preprocessing

In [ ]:
X = df.drop(columns=['capacity_remaining_pct'])
y = df['capacity_remaining_pct']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train samples:', X_train.shape[0])
print('Test  samples:', X_test.shape[0])

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Scaling done.')
print('Mean (train):', X_train_sc.mean(axis=0).round(3))
print('Std  (train):', X_train_sc.std(axis=0).round(3))

### 5. Build ANN Model
Architecture:
- Input layer: 7 features
- Hidden layers: Dense(128) -> Dense(256) -> Dense(128) -> Dense(64) -> Dense(32)
- Each hidden layer uses ReLU activation with BatchNormalization and Dropout
- Output layer: Dense(1) with linear activation for regression

In [ ]:
tf.random.set_seed(42)

model = keras.Sequential([
    layers.Input(shape=(X_train_sc.shape[1],)),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.1),

    layers.Dense(32, activation='relu'),

    layers.Dense(1, activation='linear')
], name='battery_ann')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

model.summary()

### 6. Train the Model

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=25, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1
    )
]

history = model.fit(
    X_train_sc, y_train,
    validation_split=0.15,
    epochs=300,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

### 7. Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history['loss'],     label='train loss')
axes[0].plot(history.history['val_loss'], label='val loss')
axes[0].set_title('Loss (MSE) over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'],     label='train MAE')
axes[1].plot(history.history['val_mae'], label='val MAE')
axes[1].set_title('MAE over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Best epoch: {len(history.history["loss"])}')

### 8. Model Evaluation on Test Set

In [ ]:
y_pred = model.predict(X_test_sc).flatten()

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print('--- Test Metrics ---')
print(f'MSE  : {mse:.4f}')
print(f'RMSE : {rmse:.4f} %')
print(f'MAE  : {mae:.4f} %')
print(f'R2   : {r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# actual vs predicted
mn, mx = y_test.min(), y_test.max()
axes[0].scatter(y_test, y_pred, alpha=0.4, s=12, color='darkcyan')
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='perfect fit')
axes[0].set_xlabel('Actual Capacity (%)')
axes[0].set_ylabel('Predicted Capacity (%)')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# residuals
residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.4, s=12, color='salmon')
axes[1].axhline(0, color='black', linewidth=1.2, linestyle='--')
axes[1].set_xlabel('Predicted Capacity (%)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(residuals, bins=40, color='slateblue', edgecolor='white')
plt.axvline(0, color='red', linestyle='--', linewidth=1.5)
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Residual Distribution')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 9. Feature Importance (via Permutation)
Estimate which features matter most by measuring how much the RMSE increases when each feature is shuffled.

In [ ]:
base_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
importances = {}

for i, feat in enumerate(X.columns):
    X_perm = X_test_sc.copy()
    np.random.shuffle(X_perm[:, i])
    perm_pred = model.predict(X_perm, verbose=0).flatten()
    perm_rmse = np.sqrt(mean_squared_error(y_test, perm_pred))
    importances[feat] = perm_rmse - base_rmse

imp_df = pd.DataFrame.from_dict(importances, orient='index', columns=['importance_increase_rmse'])
imp_df = imp_df.sort_values('importance_increase_rmse', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(imp_df.index, imp_df['importance_increase_rmse'], color='steelblue', edgecolor='white')
plt.xlabel('Increase in RMSE when feature is shuffled')
plt.title('Permutation Feature Importance')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print(imp_df.sort_values('importance_increase_rmse', ascending=False))

### 10. Predict on New Battery Samples

In [ ]:
new_batteries = pd.DataFrame({
    'charge_cycles'            : [50,   800,  1400],
    'avg_temperature_c'        : [25.0, 38.0, 50.0],
    'depth_of_discharge'       : [0.3,  0.7,  1.0],
    'avg_charge_rate_c'        : [0.5,  1.5,  2.8],
    'age_months'               : [3,    24,   55],
    'internal_resistance_mohm' : [12.0, 25.0, 42.0],
    'avg_discharge_current_a'  : [1.0,  2.5,  4.5]
})

new_sc   = scaler.transform(new_batteries)
new_pred = model.predict(new_sc).flatten()

new_batteries['predicted_capacity_pct'] = new_pred.round(2)

labels = ['New battery (low stress)', 'Mid-life battery', 'Aged battery (high stress)']
new_batteries.insert(0, 'description', labels)

print('Battery Health Predictions:')
new_batteries

### 11. Summary

| Item | Detail |
|---|---|
| Dataset | Custom synthetic battery degradation (1200 rows, 7 features) |
| Target | Remaining capacity as % of original |
| Task | Regression |
| Model | ANN - 5 hidden layers with BatchNorm + Dropout |
| Optimizer | Adam with ReduceLROnPlateau |
| Loss | Mean Squared Error (MSE) |
| Callbacks | EarlyStopping + ReduceLROnPlateau |
| Extra | Permutation feature importance analysis |